# 🧬 Demo: Single-Cell RNA-seq Analysis Workflow

This notebook demonstrates a complete scRNA-seq analysis workflow using the analysis template.

**Goals:**
1. Show how to use the local analysis package
2. Demonstrate which steps to run locally vs. on a GPU cluster (Euler)
3. Illustrate the local ↔ remote sync workflow via Git/GitHub

**Legend:**
- 💻 **Local** — Fast, good for development, code editing
- 🚀 **GPU/Euler** — Heavy compute, model fitting, GPU-accelerated

**Changelog:**
- 2026-01-27: Initial demo notebook

## Preliminaries

### Dependency notebooks

This is a standalone demo — no dependencies on other notebooks.

### Library imports

`autoreload` to re-load packages.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import matplotlib.pyplot as plt
import scanpy as sc

# Local analysis package — edit these functions in src/myanalysis/
from myanalysis import FilePaths, qc_violin

warnings.filterwarnings("ignore", category=FutureWarning)

### General settings

In [ ]:
sc.settings.verbosity = 2
sc.settings.datasetdir = FilePaths.EXAMPLE_DATASET / "raw"
sc.settings.set_figure_params(dpi=100, frameon=False)
sc.settings.figdir = FilePaths.FIGURES / "example_dataset"

print(f"Project root: {FilePaths.ROOT}")
print(f"Data folder:  {FilePaths.DATA}")

### Function definitions

Any utility functions specific to this notebook go here. For reusable functions, add them to `src/myanalysis/`.

### Data loading

We'll use the classic PBMC 3k dataset from 10X Genomics, available via scanpy.

In [ ]:
# Download PBMC 3k (cached after first download)
adata = sc.datasets.pbmc3k()
adata

In [ ]:
# Store raw counts for scVI (needs raw integer counts)
adata.layers["counts"] = adata.X.copy()

---

## Main analysis

### 🔬 Quality Control — 💻 Local

QC is fast and benefits from quick iteration — perfect for local development.

In [ ]:
# Annotate mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

In [ ]:
# Use our custom plotting function from the local package!
# 💡 TIP: Edit src/myanalysis/plotting.py locally, then run again, autoreload will update automatically!
fig = qc_violin(adata, figsize=(12, 3))
plt.show()

In [ ]:
# Filter cells and genes
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f"After filtering:  {adata.n_obs} cells, {adata.n_vars} genes")

### 🧮 Preprocessing — 💻 Local

Normalization and HVG selection are fast operations.

In [ ]:
# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Identify highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

### 🚀 scVI Model Training — 🚀 GPU/Euler

**This section benefits from GPU acceleration!**

Workflow for running on Euler:
1. 💻 Commit & push your notebook: `git add . && git commit -m "Ready for scVI" && git push`
2. 🖥️ SSH to Euler, pull changes: `git pull`
3. 🚀 Run this section in JupyterHub: https://jupyter.euler.hpc.ethz.ch
4. 💾 Save results and push: `git add . && git commit -m "scVI trained" && git push`
5. 💻 Pull results locally: `git pull`

In [ ]:
import scvi

# Check if GPU is available
import torch

print(f"PyTorch device: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")
print(f"MPS available (Apple Silicon): {torch.backends.mps.is_available()}")

In [ ]:
# Setup scVI model
scvi.model.SCVI.setup_anndata(adata, layer="counts")

# Create and train model
# 💡 On GPU: ~1 min | On CPU: ~5-10 min
model = scvi.model.SCVI(adata, n_latent=10, n_layers=2)
model.train(max_epochs=10, early_stopping=True)

In [ ]:
# Get latent representation
adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"scVI latent shape: {adata.obsm['X_scVI'].shape}")

### ⚡ Neighbors & UMAP — 🚀 GPU/Euler (rapids-singlecell)

On Linux with NVIDIA GPU, we can use `rapids-singlecell` for massive speedups.
Falls back to scanpy on macOS.

In [ ]:
import sys

# Use rapids-singlecell on Linux (GPU), scanpy on macOS
if sys.platform == "linux":
    try:
        import rapids_singlecell as rsc

        print("✅ Using rapids-singlecell (GPU-accelerated)")
        USE_RAPIDS = True
    except ImportError:
        print("⚠️ rapids-singlecell not available, falling back to scanpy")
        USE_RAPIDS = False
else:
    print("💻 macOS detected, using scanpy (CPU)")
    USE_RAPIDS = False

In [ ]:
# Compute neighbors and UMAP
# 💡 With rapids on GPU: seconds | With scanpy on CPU: ~30s for 3k cells

if USE_RAPIDS:
    # GPU-accelerated
    rsc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
    rsc.tl.umap(adata)
else:
    # CPU fallback
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata)

In [ ]:
# Clustering (Leiden)
if USE_RAPIDS:
    rsc.tl.leiden(adata, resolution=0.5, key_added="leiden")
else:
    sc.tl.leiden(adata, resolution=0.5, key_added="leiden", flavor="igraph", n_iterations=2)

print(f"Found {adata.obs['leiden'].nunique()} clusters")

### 🎨 Visualization — 💻 Local

Back to local for visualization and figure generation.

In [ ]:
sc.pl.embedding(adata, basis="umap", color=["leiden", "CST3", "NKG7", "PPBP", "CD8A"], ncols=5, save="_overview.png")

### 🏷️ Cell Type Annotation — 💻 Local

Let's first get the model, if we haven't already downloaded it. 

In [ ]:
import celltypist
from celltypist import models

models.download_models(model="Immune_All_High.pkl")

Now we use the downloaded model for annotation.

⚠️ **Input scale matters.** CellTypist's `Immune_All_High` model expects expression that is **normalized to 10,000 counts per cell and log1p-transformed** (the same transform we applied in the preprocessing step). Annotating data on a different scale (e.g. raw counts, or `sc.pp.scale`-d/z-scored values) silently produces wrong labels. To stay robust we rebuild the expected input from the raw `counts` layer rather than assuming `adata.X` is still in that state.

In [ ]:
# Annotate on a fresh copy whose .X is guaranteed to be log1p(normalize_total(1e4)),
# rebuilt from the raw counts — independent of any later transforms on `adata.X`.
adata_celltypist = adata.copy()
adata_celltypist.X = adata_celltypist.layers["counts"].copy()
sc.pp.normalize_total(adata_celltypist, target_sum=1e4)
sc.pp.log1p(adata_celltypist)

model = models.Model.load(model="Immune_All_High.pkl")
predictions = celltypist.annotate(adata_celltypist, model=model, majority_voting=True)

In [ ]:
# Add to adata
adata.obs["cell_type"] = predictions.predicted_labels["majority_voting"]
adata.obs["cell_type"].value_counts()

Visualize the umap again. 

In [ ]:
sc.pl.embedding(
    adata,
    basis="umap",
    color=[
        "leiden",
        "CST3",
        "NKG7",
        "PPBP",
        "CD8A",
        "cell_type",
    ],
)

### 💾 Save Results — 💻 Local

Save processed data following the template's data organization.

In [ ]:
output_path = FilePaths.EXAMPLE_DATASET / "processed" / "pbmc3k_processed.h5ad"
adata.write(output_path)
print(f"Saved to: {output_path}")

---

### 📋 Session Info

In [ ]:
from session_info2 import session_info

session_info()

### 🔄 Workflow Summary


**Git sync workflow:**
```bash
# Local: push changes
git add . && git commit -m "description" && git push

# Euler: pull and run
git pull
# ... run heavy compute ...
git add . && git commit -m "results" && git push

# Local: pull results
git pull
```